## tuoitre.vn

In [3]:
import os
import requests
from bs4 import BeautifulSoup, NavigableString
from tqdm import tqdm

In [81]:
url = 'https://tuoitre.vn/nan-quay-roi-tinh-duc-trong-quan-doi-nhat-20240513111547293.htm'
response = requests.get(url).content
soup = BeautifulSoup(response, 'html.parser')

In [4]:
def get_text_from_tag(tag):
    if isinstance(tag, NavigableString):
        return tag
                    
    # else if isinstance(tag, Tag):
    return tag.text;

In [5]:
def extract_content(url):
    """
    Extract title, description and paragraphs from url
    @param url (str): url to crawl
    @return title (str)
    @return description (generator)
    @return paragraphs (generator)
    """
    content = requests.get(url).content
    soup = BeautifulSoup(content, "html.parser")

    title = soup.find('h1', 'detail-title article-title')
    if title is None:
        title = soup.find('h1', 'detail-title')
        
    if title == None:
        return None, None, None
    title = title.text

    # some sport news have location-stamp child tag inside description tag
    description = (get_text_from_tag(p) for p in soup.find("h2", class_="detail-sapo").contents)
    if not description:
        return None, None, None

    content_div = soup.find('div', class_='detail-content afcbc-body')
    # Extract and join the text from all <p> tags & avoid get the text from IMAGES 
    paragraphs = (get_text_from_tag(child) for child in content_div.children if child.name=='p')
    if not paragraphs:
        return None, None, None

    return title, description, paragraphs

In [6]:
def write_content(url, output_fpath, failed_urls):
    """
    From url, extract title, description and paragraphs then write in output_fpath
    @param url (str): url to crawl
    @param output_fpath (str): file path to save crawled result
    @return (bool): True if crawl successfully and otherwise
    """
    title, description, paragraphs = extract_content(url)
                
    if not title or not description or not paragraphs:
        print('fail')
        failed_urls.append(url)
        return False

    with open(output_fpath, "w", encoding="utf-8") as file:
        file.write(title + "\n\n")
        for p in description:
            file.write(p.strip() + "\n\n")
        for p in paragraphs:                     
            file.write(p + "\n")

In [170]:
url = 'https://tuoitre.vn/tranh-cai-vu-canh-sat-my-nghi-ban-nham-ha-si-khong-quan-20240510180754432.htm'
title, description, paragraphs = extract_content(url)

In [171]:
title

'Tranh cãi vụ cảnh sát Mỹ nghi bắn nhầm hạ sĩ không quân'

In [182]:
print(next(iter(paragraphs)))

Luật sư Ben Crump, người đại diện gia đình anh Fortson, chỉ ra việc phía cảnh sát nhận trình báo về tiếng ồn đánh nhau mâu thuẫn với việc nạn nhân ở nhà một mình vào thời điểm diễn ra vụ việc.


In [185]:
failed_urls

['https://raovat.tuoitre.vn/nha-dat/dat-nha-ngang-12m-mat-tien-duong-lon-30m-4-lan-o-to-chay-57863',
 'https://raovat.tuoitre.vn/nha-dat/ban-lo-70m2-4mx175m-cach-nga-4-vo-van-kiet-chi-900m-duong-vao-20m-58139',
 'https://raovat.tuoitre.vn/nha-dat/cho-thue-nha-mat-tien-2-can-phuong-tan-thanh-quan-tan-phu-57995',
 'https://raovat.tuoitre.vn/nha-dat/ban-nha-mat-tien-cho-binh-trieuphuong-hiep-binh-chanh-tp-thu-duc-58112',
 'https://raovat.tuoitre.vn/nha-dat/70m2-dat-kdc-hien-huu-cach-truong-ngo-thoi-nhiem-1kmthong-vo-van-kiet-58176',
 'https://raovat.tuoitre.vn/nha-dat/sang-dat-xa-long-hoa-huyen-can-gio-dat-bien-tho-cu-100-55981',
 'https://tuoitre.vn/trung-quoc-yeu-cau-tham-van-voi-canada-tai-wto-vi-muc-thue-khung-voi-xe-dien-nhom-thep-20240906194237512.htm',
 'https://tuoitre.vn/mot-viet-nam-moi-sau-30-nam-hoi-nhap-20240829102115184.htm',
 'https://tuoitre.vn/my-huy-thoa-thuan-tranh-an-tu-voi-nghi-pham-vu-khung-bo-11-9-20240803084520824.htm',
 'https://tuoitre.vn/truc-thang-quan-su-mi-28

In [184]:
the_gioi_parent_path = '/Users/vinbrain/Library/CloudStorage/OneDrive-TAPDOANVINGROUP/VNExpressCrawler/data_tuoi-tre/the-gioi/'
the_gioi_url_path = os.path.join(the_gioi_parent_path, 'the-gioi_article_urls.txt')
# failed_urls = []
with open(the_gioi_url_path, 'r') as file:
    lines = file.readlines()
    for i, url in tqdm(enumerate(lines[3727:]), total=len(lines[3727:])):
        url = url.strip()
        if 'https://raovat' in url:
            failed_urls.append(url)
            continue
        index = i+1
        content_name = f'url_{index}.txt'
        content_path = os.path.join(the_gioi_parent_path, 'contents', content_name)
        write_content(url, content_path, failed_urls)

 36%|███▋      | 180/494 [00:44<01:11,  4.38it/s]

fail


 61%|██████▏   | 303/494 [01:21<01:06,  2.86it/s]

fail


100%|██████████| 494/494 [02:11<00:00,  3.76it/s]


In [7]:
phap_luat_parent_path = '/Users/vinbrain/Library/CloudStorage/OneDrive-TAPDOANVINGROUP/VNExpressCrawler/data_tuoi-tre/phap-luat/'
phap_luat_url_path = os.path.join(phap_luat_parent_path, 'phap-luat_article_urls.txt')
phap_luat_failed_urls = []
with open(phap_luat_url_path, 'r') as file:
    lines = file.readlines()
    for i, url in tqdm(enumerate(lines), total=len(lines)):
        url = url.strip()
        if 'https://raovat' in url:
            phap_luat_failed_urls.append(url)
            continue
        index = i+1
        content_name = f'url_{index}.txt'
        content_path = os.path.join(phap_luat_parent_path, 'contents', content_name)
        write_content(url, content_path, phap_luat_failed_urls)

  5%|▌         | 309/5823 [00:46<50:48,  1.81it/s]  

fail


  8%|▊         | 440/5823 [01:04<14:09,  6.33it/s]

fail


 13%|█▎        | 732/5823 [01:46<16:17,  5.21it/s]

...

 92%|█████████▏| 5330/5823 [15:25<01:29,  5.50it/s]

fail


 92%|█████████▏| 5355/5823 [15:36<01:09,  6.69it/s]

fail


100%|██████████| 5823/5823 [16:46<00:00,  5.78it/s]


In [9]:
kinh_doanh_parent_path = '/Users/vinbrain/Library/CloudStorage/OneDrive-TAPDOANVINGROUP/VNExpressCrawler/data_tuoi-tre/kinh-doanh/'
kinh_doanh_url_path = os.path.join(kinh_doanh_parent_path, 'kinh-doanh_article_urls.txt')
kinh_doanh_failed_urls = []
with open(kinh_doanh_url_path, 'r') as file:
    lines = file.readlines()
    for i, url in tqdm(enumerate(lines), total=len(lines)):
        url = url.strip()
        if 'https://raovat' in url:
            kinh_doanh_failed_urls.append(url)
            continue
        index = i+1
        content_name = f'url_{index}.txt'
        content_path = os.path.join(kinh_doanh_parent_path, 'contents', content_name)
        write_content(url, content_path, kinh_doanh_failed_urls)

  1%|▏         | 60/4267 [00:07<22:42,  3.09it/s]

fail


 12%|█▏        | 503/4267 [01:04<07:46,  8.07it/s]

fail


 14%|█▍        | 597/4267 [01:16<08:50,  6.92it/s]

...

fail


 43%|████▎     | 1855/4267 [04:13<06:52,  5.85it/s]

fail


 46%|████▌     | 1944/4267 [04:25<05:16,  7.33it/s]


ChunkedEncodingError: ('Connection broken: IncompleteRead(32455 bytes read, 23200 more expected)', IncompleteRead(32455 bytes read, 23200 more expected))